In [ ]:
!git clone https://github.com/ultralytics/yolov5  # Clone YoloV5 repository
%cd yolov5
!pip install -r requirements.txt  # Install dependencies

Cloning into 'yolov5'...
remote: Enumerating objects: 16965, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 16965 (delta 79), reused 102 (delta 50), pack-reused 16805 (from 1)
Receiving objects: 100% (16965/16965), 15.71 MiB | 12.32 MiB/s, done.
Resolving deltas: 100% (11617/11617), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.1/874.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Replace with your Roboflow dataset link
!curl -L "https://app.roboflow.com/ds/1ebM0ElsMc?key=3zEJKWkGGI" > roboflow.zip
!unzip roboflow.zip -d roboflow


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   903  100   903    0     0   2844      0 --:--:-- --:--:-- --:--:--  2848
100 41.3M  100 41.3M    0     0  23.8M      0  0:00:01  0:00:01 --:--:-- 33.7M
Archive:  roboflow.zip
  inflating: roboflow/README.dataset.txt  
  inflating: roboflow/README.roboflow.txt  
   creating: roboflow/test/
   creating: roboflow/test/Above T1/
 extracting: roboflow/test/Above T1/Above-T1-10-_png.rf.0679e1b5fdd5beabfab5cca7f23ea002.jpg  
 extracting: roboflow/test/Above T1/Above-T1-102-_png.rf.e64f80a684bf08d6e3abf5f00b43f429.jpg  
 extracting: roboflow/test/Above T1/Above-T1-107-_png.rf.f7379362530003bb87c57399ddc2ffd8.jpg  
 extracting: roboflow/test/Above T1/Above-T1-109-_png.rf.d4397625d0d2545592fa868b4a0f46de.jpg  
 extracting: roboflow/test/Above T1/Above-T1-110-_png.rf.1dae576fd5cd1119773b92ffc197f59c.jpg  
 extracting: roboflow/test/Abo

In [ ]:
train: '/content/drive/MyDrive/BLC Dataset/SAVE_split/train'  # Update with correct path
val: '/content/drive/MyDrive/BLC Dataset/SAVE_split/test'    # Update with correct path

nc: 6  # Number of classes
names: ['TURBT', 'TA', 'Normal', 'Loop', 'Above T1', 'Hose']# Class names


In [ ]:
# Path to your dataset directory
train_data = '/content/drive/MyDrive/BLC Dataset/SAVE_split'
val_data = '/content/drive/MyDrive/BLC Dataset/SAVE_split'

# Train the classification model
!python /content/yolov5/classify/train.py --model yolov5s-cls.pt --data {train_data} --epochs 100 --batch-size 16 --img 224 --workers 4


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
2024-09-24 03:24:54.674415: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-24 03:24:54.825548: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-24 03:24:54.869168: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
classify/train: model=yolov5s-cls.pt, data=/content/yo

In [ ]:
!python classify/val.py --weights /content/yolov5/runs/train-cls/exp/weights/best.pt --data '/content/yolov5/roboflow' --img-size 224 --batch-size 16


classify/val: data=/content/yolov5/roboflow, weights=['/content/yolov5/runs/train-cls/exp/weights/best.pt'], batch_size=16, imgsz=224, device=, workers=8, verbose=True, project=runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-368-gb163ff8d Python-3.10.12 torch-2.4.1+cu121 CUDA:0 (Tesla T4, 15102MiB)

Fusing layers... 
Model summary: 117 layers, 4174374 parameters, 0 gradients, 10.4 GFLOPs
testing:   0% 0/16 [00:00<?, ?it/s]/content/yolov5/classify/val.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
testing: 100% 16/16 [00:01<00:00, 15.03it/s]
                   Class      Images    top1_acc    top5_acc
                     all         248       0.984           1
                Above T1          32       0.938           1
                    Hose          29           1           1
                    Loop          4

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load true and predicted labels
y_true = np.loadtxt('y_true.txt', dtype=int)
y_pred = np.loadtxt('y_pred.txt', dtype=int)

# Generate classification report
target_names = ['TURBT', 'TA', 'Normal', 'Loop', 'Above T1', 'Hose']
print(classification_report(y_true, y_pred, target_names=target_names))

# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()


FileNotFoundError: y_true.txt not found.